# <center> 1. Librerias

In [1]:
import sys
from pathlib import Path

# 1. Encontrar la raíz del proyecto (subir un nivel desde la carpeta 'notebooks')
# Si tu notebook está en la raíz, usá Path(".").resolve()
ROOT_DIR = Path("..").resolve()

# 2. Agregar la raíz al sistema de Python si no está cargada
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
import os    
import pandas as pd
import numpy as np
import openpyxl
from src.preprocessing.ipc import format_df_IPC
from src.preprocessing.common import format_df, save_csv
from src.paths import (
    AUH_DIR,
    CBTA_DIR,
    IPC_DIR,
    EPH_DIR,
    ENGHO_DIR,
    INTERIM_DIR,
)
%load_ext autoreload
%autoreload 2

# <center> 2. Carga y Ajuste de Estructura

## <center> Cargamos AUH

In [2]:
hojas_auh = pd.read_excel(AUH_DIR/"AUH_data.xlsx", sheet_name=None)
hojas_auh

{'Boletín ': Empty DataFrame
 Columns: []
 Index: [],
 'Contenido ':                                            Unnamed: 0  \
 0   Boletín mensual de Asignaciones para Protecció...   
 1                                     Febrero de 2026   
 2                                                 NaN   
 3                                                 NaN   
 4                                           Contenido   
 5                                                 NaN   
 6                                    1. Beneficiarios   
 7                                                 1.1   
 8                                               1.1.A   
 9                                               1.1.B   
 10                                                1.2   
 11                                              1.2.A   
 12                                              1.2.B   
 13                                                1.3   
 14                                                1.4   
 15 

In [3]:
nombres_hojas = list(hojas_auh.keys())
print(f"Hojas encontradas en el archivo de AUH: {nombres_hojas}\n")


Hojas encontradas en el archivo de AUH: ['Boletín ', 'Contenido ', 'Consideraciones - Definiciones', '1.1 Benef. H Edad', '1.1.A Benef. H Muj Edad ', '1.1.B Benef. H Var Edad', '1.2 Benef. HD Edad', '1.2.A Benef. HD Muj Edad', '1.2.B Benef. HD Var Edad', '1.3 Benef. Ciclo Lect', '1.4 Benef. Situ. Mayor', '1.5 SUAF x sexo', '1.6 Población ', '1.7 Apertura otras incompatib', '1.8 Benef AxE', '1.9 Benef Apoyo Alimentario', '2.1 Titu. Sexo', '2.2 Titu Edad', '2.3 Hijos por Titu', '2.4 Titu Apoyo Alimentario', '3.1 Importes Liq.', '3.2 Importes Liq. SUAF', '3.3 Importes Liq AxE', '3.4 Importes Liq. Apoyo Alim', '3.5 Valor general pago', '3.6 Valor general-$ Feb 26', '4.1 Corresponsabilidades', '5.1 Indicador', '6.1 Tiempo Prom Cob', '6.2 Benef AxE forma acceso', '6.3 Corresponsabilidad automáti']



## Vemos que tenemos multiples hojas, nos quedamos solo con las que nos hagan falta.

### Valores generales de la AUH:

In [4]:
hojas_auh['3.5 Valor general pago'].head()

,"Cuadro 3.5 - Valor general, en pesos corrientes, de la Asignación Universal por Hijo e Hijo con Discapacidad, de la Asignación por embarazo y del Apoyo Alimentario - Ley 1000 días",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Volver al índice
0,NaN,NaN,NaN,NaN,NaN,NaN
1,Periodo,Hijo,Hijo con Discapacidad,Asignación por Embarazo,Apoyo Alimentario - Ley 1000 días,NaN
2,2009-10-01 00:00:00,180,720,NaN,NaN,NaN
3,2010-10-01 00:00:00,220,880,NaN,NaN,NaN
4,2011-05-01 00:00:00,220,880,220,NaN,NaN


### Arreglemos la estructura:

In [5]:
df_AUH_valor_general = format_df('3.5 Valor general pago', hojas_auh, 2, -4, 1)

In [6]:
df_AUH_valor_general.head()

1,Periodo,Hijo,Hijo con Discapacidad,Asignación por Embarazo,Apoyo Alimentario - Ley 1000 días
0,2009-10-01 00:00:00,180,720,NaN,NaN
1,2010-10-01 00:00:00,220,880,NaN,NaN
2,2011-05-01 00:00:00,220,880,220,NaN
3,2011-09-01 00:00:00,270,1080,270,NaN
4,2012-09-01 00:00:00,340,1200,340,NaN


### Edades de los titulares del beneficio:

In [7]:
hojas_auh['2.2 Titu Edad'].tail()

,"Cuadro 2.2 - Titulares de la Asignación Universal por Hijo por mes, según grupo de edad",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Volver al índice
36,2026-02-01 00:00:00,47631,250707,469085,536368,436850,309353,200084,88102,28329,4757,536,236,4721,2376759,34.257115,NaN
37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38,Nota: el periodo corresponde al mes de liquida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40,Fuente: ANSES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Arreglemos la estructura:

In [8]:
# 1. Seleccionamos las filas 1 y 2 (la fila 0 está vacía, no aporta datos)
encabezados = hojas_auh['2.2 Titu Edad'].iloc[1:3]

# 2. Fusionamos la fila 2 (rangos de edad) rellenando sus NaN con la fila 1 (Periodo, Total, etc.)
nuevas_columnas = encabezados.iloc[1].fillna(encabezados.iloc[0])

# 3. Asignamos los nombres resultantes como las columnas oficiales del DataFrame
hojas_auh['2.2 Titu Edad'].columns = nuevas_columnas

# 4. Eliminamos las filas de encabezado (0, 1 y 2) para dejar solo los datos numéricos abajo
df_AUH_edad_titular = hojas_auh['2.2 Titu Edad'].iloc[3:-4].reset_index(drop=True)
df_AUH_edad_titular = df_AUH_edad_titular.loc[:, df_AUH_edad_titular.columns.notna()]

In [9]:
df_AUH_edad_titular.tail()

2,Periodo,15 - 19,20 - 24,25 - 29,30 - 34,35 - 39,40 - 44,45 - 49,50 - 54,55 - 59,60 - 64,65 - 69,Más de 70,Sin datos,Total,Edad Promedio
29,2025-10-01 00:00:00,49257,256974,474493,530194,428984,305007,195810,86289,27387,4434,510,232,4400,2363971,34.128514
30,2025-11-01 00:00:00,48826,255570,473359,531912,431046,305653,197222,86771,27654,4529,494,233,4489,2367758,34.160209
31,2025-12-01 00:00:00,48429,254085,472109,534062,433524,307079,198770,87309,27994,4573,520,231,4587,2373272,34.196781
32,2026-01-01 00:00:00,47981,252126,470424,534868,434918,308168,199169,87617,28102,4622,518,233,4570,2373316,34.225823
33,2026-02-01 00:00:00,47631,250707,469085,536368,436850,309353,200084,88102,28329,4757,536,236,4721,2376759,34.257115


##  <center>Cargamos CBT Y CBA

In [10]:
CBA = pd.read_csv(CBTA_DIR/"canasta-basica-alimentaria-regiones-del-pais.csv")
CBT = pd.read_csv(CBTA_DIR/"canasta-basica-total-regiones-del-pais.csv")
pobreza_indigencia = pd.read_csv(CBTA_DIR/"valores-canasta-basica-alimentos-canasta-basica-total-mensual-2016.csv")

### Veamos los datos de la canasta basica alimentaria:

In [12]:
CBA.head()

,indice_tiempo,gran_buenos_aires,cuyo,noreste,noroeste,pampeana,patagonia
0,2016-04-01,1514.53,1358.29,1371.62,1333.91,1514.96,1556.96
1,2016-05-01,1561.35,1398.20,1404.27,1369.37,1562.75,1604.30
2,2016-06-01,1614.32,1445.22,1447.63,1413.77,1614.33,1661.42
3,2016-07-01,1666.48,1494.04,1496.21,1458.24,1660.19,1713.67
4,2016-08-01,1675.05,1496.09,1501.91,1459.38,1662.99,1723.86


In [13]:
CBA.tail()

,indice_tiempo,gran_buenos_aires,cuyo,noreste,noroeste,pampeana,patagonia
112,2025-08-01,168456.04,150019.85,149802.94,145932.54,165643.09,172540.84
113,2025-09-01,170788.46,152803.20,152566.11,148800.44,168519.55,175412.47
114,2025-10-01,176150.29,157636.84,157679.95,153475.78,174130.91,181342.77
115,2025-11-01,183289.46,163650.50,164584.23,159191.14,181196.12,189380.65
116,2025-12-01,190780.09,169325.29,171065.04,164555.21,188144.43,197032.47


In [14]:
CBT.head()

,indice_tiempo,gran_buenos_aires,cuyo,noreste,noroeste,pampeana,patagonia
0,2016-04-01,3665.17,3504.40,3099.86,2987.95,3666.21,4281.63
1,2016-05-01,3825.30,3649.31,3229.82,3122.17,3828.74,4459.96
2,2016-06-01,3938.94,3757.58,3315.07,3209.27,3938.96,4602.13
3,2016-07-01,4032.88,3854.62,3396.40,3281.04,4017.66,4712.59
4,2016-08-01,4036.87,3844.95,3394.32,3269.01,4007.81,4723.38


In [15]:
CBT.tail()

,indice_tiempo,gran_buenos_aires,cuyo,noreste,noroeste,pampeana,patagonia
112,2025-08-01,375656.97,355547.04,313088.14,302080.36,369384.09,436528.33
113,2025-09-01,380858.27,363671.62,318863.17,308016.91,375798.60,445547.67
114,2025-10-01,392815.15,375175.68,329551.10,317694.86,388311.93,458797.21
115,2025-11-01,406902.60,387851.69,342335.20,327933.75,402255.39,477239.24
116,2025-12-01,423531.80,399607.68,355815.28,338983.73,417680.63,496521.82


In [11]:
pobreza_indigencia

,indice_tiempo,canasta_basica_alimentaria,inversa_coeficiente_engel,canasta_basica_total,linea_indigencia,linea_pobreza
0,2016-04-01,1514.53,2.42,3663.66,4679.9100,1.132071e+04
1,2016-05-01,1561.35,2.45,3830.77,4824.5600,1.183708e+04
2,2016-06-01,1614.32,2.44,3942.67,4988.2500,1.218284e+04
3,2016-07-01,1666.48,2.42,4033.76,5149.4100,1.246433e+04
4,2016-08-01,1675.05,2.41,4041.87,5175.9200,1.248937e+04
...,...,...,...,...,...,...
118,2026-02-01,208442.85,2.17,452320.98,644088.4100,1.397672e+06
119,2026-03-01,212948.52,2.18,464227.77,658010.9268,1.434464e+06
120,2026-04-01,215227.62,2.21,475653.04,665053.3458,1.469768e+06
121,2026-05-01,220467.99,2.20,485029.58,681246.0891,1.498741e+06


### Estan correctos de estructura

## <center> Cargamos IPC

In [16]:
hojas_ipc = pd.read_excel(IPC_DIR/"ipc_nacional.xlsx", sheet_name=None)

In [17]:
hojas_ipc

{'Indice':                Unnamed: 1  Unnamed: 2  \
 0   NaN            INDICE         NaN   
 1   NaN               NaN         NaN   
 2   NaN  Cuadros -Precios         NaN   
 3   NaN               NaN         NaN   
 4   NaN             4.1.1         NaN   
 ..   ..               ...         ...   
 124 NaN              4.36         NaN   
 125 NaN               NaN         NaN   
 126 NaN              4.37         NaN   
 127 NaN               NaN         NaN   
 128 NaN              4.38         NaN   
 
                                             Unnamed: 3  
 0                                                  NaN  
 1                                                  NaN  
 2                                                  NaN  
 3                                                  NaN  
 4                                    IPC Nivel General  
 ..                                                 ...  
 124  Índice de precios al consumidor Tucumán. Nivel...  
 125                

In [18]:
nombres_hojas_ipc = list(hojas_ipc.keys())
print(f"Hojas encontradas en el archivo de IPC: {nombres_hojas_ipc}\n")

Hojas encontradas en el archivo de IPC: ['Indice', '4.1.1 IPC NG', '4.1.2 IPC Capitulos', 'Hoja1', '4.1.3 IPC Bs Ss', '4.1.4 IPC Incidencia Cap', '4.1.5 IPC Incidencia Bs Ss', '4.1.6 IPC Categorias', '4.1.7 Incidencia Cat', '4.1.8 IPC precios canasta', '4.2.1', '4.2.2', '4.2.3', '4.2.4', '4.2.5', '4.3 IPIM', '4.3.1 IPIM 4 dígitos', '4.4 IPIB', '4.4.1 IPIB 4 dígitos', '4.5 IPP', '4.5.1 IPP 4 dígitos', '4.6 ICC', '4.7.1', '4.7.2', '4.8.1 ', '4.8.2', '4.8.3', '4.9.1', '4.9.2', '4.9.3', '4.9.4', '4.10', '4.11', '4.12', '4.13.1', '4.13.2', '4.14', '4.15.1', '4.15.2', '4.16.2', '4.16.1', '4.16.3', '4.16.4', '4.17', '4.18', '4.19', '4.20', '4.21', '4.22', '4.23', '4.24', '4.25', '4.26', '4.27', '4.28', '4.29', '4.30', '4.31', '4.32', '4.33 San Juan', '4.34 Jujuy', '4.35 Chaco serie histórica', '4.36 Tucumán', '4.37 CABA', '4.38 Chaco']



In [19]:
hojas_ipc["4.1.1 IPC NG"].tail(20)

,Unnamed: 0,CUADRO 4.1.1.,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Índice,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
202,2025-07-01 00:00:00,9023.973,9016.5562,9016.3238,8905.132,9073.9428,9074.6511,9141.623,NaN,0.019017,...,0.021419,NaN,2025-07-01 00:00:00,0.172857,0.174218,0.172889,0.150942,0.176426,0.165109,0.184595
203,2025-08-01 00:00:00,9193.2441,9185.3068,9182.3487,9055.1527,9252.4989,9266.2277,9320.3353,NaN,0.018758,...,0.019549,NaN,2025-08-01 00:00:00,0.194858,0.196194,0.194487,0.170331,0.199576,0.189706,0.207753
204,2025-09-01 00:00:00,9384.0922,9377.7681,9366.761,9217.6682,9455.5666,9465.5075,9543.6369,NaN,0.02076,...,0.023959,NaN,2025-09-01 00:00:00,0.219662,0.221258,0.218476,0.191336,0.225903,0.215292,0.236689
205,2025-10-01 00:00:00,9603.8623,9602.5137,9584.145,9419.0224,9657.7445,9687.3328,9774.0954,NaN,0.023419,...,0.024148,NaN,2025-10-01 00:00:00,0.248226,0.250527,0.246754,0.21736,0.252116,0.243772,0.266552
206,2025-11-01 00:00:00,9841.3581,9840.8457,9821.3638,9642.9016,9879.6088,9960.8437,10001.4949,NaN,0.024729,...,0.023266,NaN,2025-11-01 00:00:00,0.279094,0.281565,0.277613,0.246295,0.28088,0.278889,0.296019
207,2025-12-01 00:00:00,10121.3715,10115.6707,10107.6074,9968.3824,10137.067,10255.4297,10258.6285,NaN,0.028453,...,0.02571,NaN,2025-12-01 00:00:00,0.315488,0.317355,0.314849,0.288362,0.314259,0.316711,0.329339
208,2026-01-01 00:00:00,10413.0309,10395.7458,10400.4968,10350.7115,10425.8507,10561.6315,10554.0348,NaN,0.028816,...,0.028796,NaN,2026-01-01 00:00:00,0.028816,0.027687,0.028977,0.038354,0.028488,0.029858,0.028796
209,2026-02-01 00:00:00,10714.6255,10667.6896,10714.2389,10676.7516,10795.0756,10921.6567,10868.5551,NaN,0.028963,...,0.029801,NaN,2026-02-01 00:00:00,0.058614,0.054571,0.060017,0.071062,0.064911,0.064963,0.059455
210,2026-03-01 00:00:00,11077.0608,11031.5404,11065.4568,11119.3596,11221.7466,11271.7383,11137.8047,NaN,0.033826,...,0.024773,NaN,2026-03-01 00:00:00,0.094423,0.09054,0.094765,0.115463,0.107001,0.0991,0.085701
211,2026-04-01 00:00:00,11363.0904,11336.5576,11330.2697,11423.3251,11506.6342,11511.107,11429.5989,NaN,0.025822,...,0.026199,NaN,2026-04-01 00:00:00,0.122683,0.120693,0.120965,0.145956,0.135105,0.12244,0.114145


In [20]:
df_ipc_general = hojas_ipc["4.1.1 IPC NG"].iloc[:, :7].copy()
df_ipc_general.columns = df_ipc_general.iloc[4]
# 2. Cortar el DataFrame para dejar solo los datos (de la fila 2 en adelante)
df_ipc_general = df_ipc_general.iloc[4:-8].reset_index(drop=True)
df_final = df_ipc_general.loc[:, df_ipc_general.columns.notna()]
df_final

4,Período,NACIONAL,GBA,PAMPEANA,NEA,NOA,CUYO
0,Período,NACIONAL,GBA,PAMPEANA,NEA,NOA,CUYO
1,145.1_INDICE_TIEMPO_DICI_T_13,145.1_IPC_NG_NACNAL_DICI_T_15,145.1_IPC_NG_GBAGBA_DICI_T_10,145.1_IPC_NG_PAMANA_DICI_T_15,145.1_IPC_NG_NEANEA_DICI_T_10,145.1_IPC_NG_NOANOA_DICI_T_10,145.1_IPC_NG_CUYUYO_DICI_T_11
2,I.17,103.806467,103.794733,103.711967,103.707467,104.122633,103.632133
3,II.17,110.448167,110.505433,110.1325,110.240233,111.430333,111.063167
4,III.17,115.579667,115.829533,115.321167,114.741733,115.9341,115.721167
...,...,...,...,...,...,...,...
205,2026-02-01 00:00:00,10714.6255,10667.6896,10714.2389,10676.7516,10795.0756,10921.6567
206,2026-03-01 00:00:00,11077.0608,11031.5404,11065.4568,11119.3596,11221.7466,11271.7383
207,2026-04-01 00:00:00,11363.0904,11336.5576,11330.2697,11423.3251,11506.6342,11511.107
208,2026-05-01 00:00:00,11607.3937,11594.5499,11556.8543,11725.5958,11747.5318,11750.0933


In [21]:
df_final[45:100]

4,Período,NACIONAL,GBA,PAMPEANA,NEA,NOA,CUYO
45,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Vemos la tabla esta dividad en dos, primero estan los valore del IPC de forma trimestral (filas 0 a 44), luego desde la fila 95 hasta terminar la tabla, los valores del IPC pero de forma mensual por año. Nos quedamos por simplicidad y mayor capacidad de transformacion con los datos mensuales.

In [22]:
df_IPC_general_mensual = df_final[94:].copy()
df_IPC_general_mensual = df_IPC_general_mensual.iloc[1:].reset_index(drop=True)

In [23]:
df_IPC_general_mensual

4,Período,NACIONAL,GBA,PAMPEANA,NEA,NOA,CUYO
0,2016-12-01 00:00:00,100,100,100,100,100,100
1,2017-01-01 00:00:00,101.5859,101.313,101.7874,101.6727,101.6014,101.7074
2,2017-02-01 00:00:00,103.6859,103.8085,103.5312,103.4617,103.7115,103.2652
3,2017-03-01 00:00:00,106.1476,106.2627,105.8173,105.988,107.055,105.9238
4,2017-04-01 00:00:00,108.9667,109.0613,108.6912,108.3473,109.9626,109.4506
...,...,...,...,...,...,...,...
110,2026-02-01 00:00:00,10714.6255,10667.6896,10714.2389,10676.7516,10795.0756,10921.6567
111,2026-03-01 00:00:00,11077.0608,11031.5404,11065.4568,11119.3596,11221.7466,11271.7383
112,2026-04-01 00:00:00,11363.0904,11336.5576,11330.2697,11423.3251,11506.6342,11511.107
113,2026-05-01 00:00:00,11607.3937,11594.5499,11556.8543,11725.5958,11747.5318,11750.0933


### <center>Carguemos ademas los precios de los alimentos de la canasta basica IPC:

In [24]:
dfs_regiones = {}

# 2. Inicializar los límites del bloque de columnas
columnas_por_region = 15
bottom = 0
top = columnas_por_region

regiones = ["GBA", "PAMPEANA", "NEA", "NOA", "CUYO", "PATAGONIA"]

for region in regiones:
    # 3. Crear la clave dinámica para el diccionario
    nombre_clave = f"df_ipc_canasta_{region}"
    
    # 4. Extraer el bloque de COLUMNAS correspondiente (todas las filas ":")
    dfs_regiones[nombre_clave] = hojas_ipc["4.1.8 IPC precios canasta"].iloc[:, bottom:top].copy()
    
    # 5. Imprimir para verificar el resultado
    print(f"--- {nombre_clave} (Columnas {bottom} a {top}) ---")
    print(dfs_regiones[nombre_clave])
    
    # 6. Desplazar los límites exactamente 14 columnas hacia adelante
    bottom = top
    top += columnas_por_region

--- df_ipc_canasta_GBA (Columnas 0 a 15) ---
              Unnamed: 0                                       CUADRO 4.1.8  \
0                    NaN  Precios al consumidor de un conjunto de elemen...   
1                    NaN                                                NaN   
2                    NaN  8.1. Precios al consumidor de un conjunto de e...   
3                    NaN                                          En pesos    
4                Período                                           G. B. A.   
..                   ...                                                ...   
113  2026-04-01 00:00:00                                            4383.37   
114  2026-05-01 00:00:00                                            4471.54   
115  2026-06-01 00:00:00                                            4652.79   
116                  NaN                                                NaN   
117        Fuente: INDEC                                                NaN   

    Un

In [25]:
df_GBA= dfs_regiones["df_ipc_canasta_GBA"]
df_GBA.head()

,Unnamed: 0,CUADRO 4.1.8,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Índice,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,NaN,Precios al consumidor de un conjunto de elemen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,8.1. Precios al consumidor de un conjunto de e...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,En pesos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Período,G. B. A.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df_PAMPEANA = dfs_regiones["df_ipc_canasta_PAMPEANA"]
df_NEA = dfs_regiones["df_ipc_canasta_NEA"]
df_NOA = dfs_regiones["df_ipc_canasta_NOA"]
df_CUYO = dfs_regiones["df_ipc_canasta_CUYO"]
df_PATAGONIA = dfs_regiones["df_ipc_canasta_PATAGONIA"]

In [27]:
df_PATAGONIA[5:6]

,Unnamed: 75,Unnamed: 76,Unnamed: 77,Unnamed: 78,Unnamed: 79,Unnamed: 80,Unnamed: 81,Unnamed: 82,Unnamed: 83,Unnamed: 84,Unnamed: 85,Unnamed: 86,Unnamed: 87,Unnamed: 88,Unnamed: 89
5,NaN,Pan francés (kg),Harina de trigo común 000 ...,Arroz blanco simple (kg),Fideos secos tipo guisero ...,Carne picada común (kg),Pollo entero (kg),"Aceite de girasol (1,5 litros)",Leche fresca entera sachet ...,Huevos de gallina (docena),Papa ...,Azúcar (kg),Detergente líquido (750cc),Lavandina (1000 cc),Jabón de tocador (125 gr)


In [28]:
df_NEA[5:6]

,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44
5,NaN,Pan francés ...,Harina de trigo común 000 ...,Arroz blanco simple (kg),Fideos secos tipo guisero ...,Carne picada común (kg),Pollo entero ...,"Aceite de girasol (1,5 litros)",Leche fresca entera sachet ...,Huevos de gallina (docena),Papa (kg),Azúcar ...,Detergente líquido (750cc),Lavandina ...,Jabón de tocador (125 gr)


In [29]:
df_GBA[0:6]

,Unnamed: 0,CUADRO 4.1.8,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Índice,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,NaN,Precios al consumidor de un conjunto de elemen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,8.1. Precios al consumidor de un conjunto de e...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,En pesos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Período,G. B. A.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,Pan francés (kg),Harina de trigo común 000 ...,Arroz blanco simple (kg),Fideos secos tipo guisero ...,Carne picada común (kg),Pollo entero ...,"Aceite de girasol (1,5 litros)",Leche fresca entera sachet ...,Huevos de gallina (docena),Papa ...,Azúcar ...,Detergente líquido (750cc),Lavandina (10...,Jabón de tocador (125 gr)


In [30]:
df_NOA[5:6]

,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56,Unnamed: 57,Unnamed: 58,Unnamed: 59
5,NaN,Pan francés (kg),Harina de trigo común 000 ...,Arroz blanco simple (kg),Fideos secos tipo guisero ...,Carne picada común (kg),Pollo entero (kg),"Aceite de girasol (1,5 litros)",Leche fresca entera sachet (litro),Huevos de gallina (docena),Papa ...,Azúcar ...,Detergente líquido (750cc),Lavandina ...,Jabón de tocador (125 gr)


In [31]:
df_ipc_canasta_GBA = format_df_IPC(df_GBA,7,-2, 5, df_GBA, False)
df_ipc_canasta_NEA = format_df_IPC(df_NEA,7,-2, 5, df_ipc_canasta_GBA, True)
df_ipc_canasta_NOA = format_df_IPC(df_NOA,7,-2, 5, df_ipc_canasta_GBA, True)
df_ipc_canasta_CUYO = format_df_IPC(df_CUYO,7,-2, 5, df_ipc_canasta_GBA, True)
df_ipc_canasta_PAMPEANA = format_df_IPC(df_PAMPEANA,7,-2, 5, df_ipc_canasta_GBA, True)
df_ipc_canasta_PATAGONIA = format_df_IPC(df_PATAGONIA,7,-2, 5, df_ipc_canasta_GBA, True)

In [32]:
ipc_canasta = {
    "GBA": df_ipc_canasta_GBA,
    "NEA": df_ipc_canasta_NEA,
    "NOA": df_ipc_canasta_NOA,
    "CUYO": df_ipc_canasta_CUYO,
    "PAMPEANA": df_ipc_canasta_PAMPEANA,
    "PATAGONIA": df_ipc_canasta_PATAGONIA,
}

## <center> Cargamos EPH

In [33]:
hojas_eph = pd.read_excel(EPH_DIR/'EPH.xlsx', sheet_name=None)
nombres_hojas_eph = list(hojas_eph.keys())
print(f"Hojas encontradas en el archivo de EPH: {nombres_hojas_eph}\n")

Hojas encontradas en el archivo de EPH: ['INDICE', 'EPH', 'TA 03-', 'TA 74-03', 'TE 03-', 'TE 74-03', 'TD 03-', 'TD 74-03', 'TS 03-', 'TS 74-03', 'TSD 03-', 'TSND 03-', 'EPH-Poblaciones', 'EAHU-Poblaciones', 'EAHU-Tasas', 'Expansion EPH a TU - 1991-2010', 'Expansion EPH a TU - Desde 2010', 'EIL - Sector', 'EIL - Aglo', 'EPH - Asal', 'EPH - Asal regiones', 'CGI-2016', 'CGI VABpb', 'CGI ManodeObra', 'CGI Puestos', 'CGI CostoSalarial', 'CGI Remun', 'CGI Remun netas', 'EIM', 'EIM Nuevo', 'IS', 'IS oct16=100', 'RIPTE', 'SMVM', 'HaberMin', 'AUH', 'CBA y CBT', 'CBA y CBT 2016', 'CBA y CBT Reg', 'CBA y CBT BA', 'EPH Puntual - PeI', 'LI - Hog', 'LI - Pers', 'LP - Hog', 'LP - Pers', 'DP', 'DP Deciles', 'OEDE Total', 'OEDE Asal rama', 'OEDE Asal prov', 'OEDE Puestos Sector', 'OEDE Puestos Rama', 'OEDE Remuneraciones', 'OEDE Remuneraciones Rama', 'Puestos x Sector', 'RT x Sector', 'RN x Sector', 'CS x Sector', 'Puestos x Prov', 'RT x Prov', 'RN x Prov', 'CS x Prov', 'SalariosInd-Indices 62-67', 'S

### Cargamos los datos del Salario Minimo Vital y Movil:

In [34]:
df_SMVM_parcial = hojas_eph["SMVM"]
df_SMVM_parcial.head(12)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Índice
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Salario mínimo, vital y móvil",NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,En pesos corrientes,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,A partir de enero de 1965,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Período,|,NaN,NaN,Moneda,NaN,NaN,NaN
8,NaN,Mensual,Diario,Horario,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
df_SMVM_parcial.tail(10)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Índice
743,2026-01-01 00:00:00,341000,13640,1705,$,NaN,NaN,NaN
744,2026-02-01 00:00:00,346800,13872.0,1734.0,$,NaN,NaN,NaN
745,2026-03-01 00:00:00,352400,14096.0,1762.0,$,NaN,NaN,NaN
746,2026-04-01 00:00:00,357800,14312.0,1789.0,$,NaN,NaN,NaN
747,2026-05-01 00:00:00,363000,14520.0,1815.0,$,NaN,NaN,NaN
748,2026-06-01 00:00:00,367800,14712.0,1839.0,$,NaN,NaN,NaN
749,2026-07-01 00:00:00,372400,14896.0,1862.0,$,NaN,NaN,NaN
750,2026-08-01 00:00:00,376600,15064.0,1883.0,$,NaN,NaN,NaN
751,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
752,Fuente: Ministerio de Capital Humano.,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
df_SMVM = format_df("SMVM", hojas_eph, 8, -2, 8)
df_SMVM = df_SMVM.drop([0,1,2])
df_SMVM

8,Mensual,Diario,Horario
3,9800,392,49
4,9800,392,49
5,9800,392,49
6,9800,392,49
7,11550,462,57.75
...,...,...,...
738,357800,14312.0,1789.0
739,363000,14520.0,1815.0
740,367800,14712.0,1839.0
741,372400,14896.0,1862.0


### Cargamos los datos de la remuneración imponible promedio de los trabajadores estables (RIPTE):


In [37]:
df_RIPTE_parcial = hojas_eph["RIPTE"]
df_RIPTE_parcial.head(12)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Índice
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Remuneración imponible promedio de los trabaja...,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Total país,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,En pesos,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Período,RIPTE,NaN,NaN,NaN,NaN,NaN,NaN
9,158.1_ICE_TIEMPO_0_0_13,158.1_REPTE_0_0_5,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
df_RIPTE_parcial.tail(10)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Índice
386,2025-11-01 00:00:00,1611851.61,NaN,NaN,NaN,NaN,NaN,NaN
387,2025-12-01 00:00:00,1633547,NaN,NaN,NaN,NaN,NaN,NaN
388,2026-01-01 00:00:00,1646344.54,NaN,NaN,NaN,NaN,NaN,NaN
389,2026-02-01 00:00:00,1734357.18,NaN,NaN,NaN,NaN,NaN,NaN
390,2026-03-01 00:00:00,1775664.12,NaN,NaN,NaN,NaN,NaN,NaN
391,2026-04-01 00:00:00,1837609.35,NaN,NaN,NaN,NaN,NaN,NaN
392,2026-05-01 00:00:00,1849727.96,NaN,NaN,NaN,NaN,NaN,NaN
393,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
394,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
395,Fuente: Ministerio de Capital Humano.,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
df_RIPTE = format_df("RIPTE", hojas_eph, 10, -3, 8)
df_RIPTE

8,Período,RIPTE
0,1994-07-01 00:00:00,874.87
1,1994-08-01 00:00:00,893
2,1994-09-01 00:00:00,907.99
3,1994-10-01 00:00:00,910.75
4,1994-11-01 00:00:00,916.93
...,...,...
378,2026-01-01 00:00:00,1646344.54
379,2026-02-01 00:00:00,1734357.18
380,2026-03-01 00:00:00,1775664.12
381,2026-04-01 00:00:00,1837609.35


## <center> Carguemos ahora ENGHo

### Cargamos Hogares:

In [40]:
df_ENGHO_hogares = pd.read_csv(ENGHO_DIR/"engho2018_hogares.txt", sep='|')
df_ENGHO_hogares.head()

,id,provincia,region,subregion,trimestre,anio,pondera,cv1c04,cv1c05_a,cv1c05_b,...,qinth_t,dinth_p,dinth_r,dinth_t,qinpch_p,qinpch_r,qinpch_t,dinpch_p,dinpch_r,dinpch_t
0,478229345,2,1,1A,4,2018,276,4,2.0,3.0,...,4,5,7,7,3,4,4,6,8,8
1,824935596,2,1,1A,4,2018,271,4,3.0,3.0,...,4,7,8,8,4,5,5,7,9,9
2,549385244,2,1,1A,1,2017,264,4,2.0,3.0,...,2,2,3,3,3,4,4,5,7,8
3,64338930,2,1,1A,3,2018,245,4,3.0,3.0,...,5,9,9,10,5,5,5,10,10,10
4,635779227,2,1,1A,3,2018,283,4,3.0,3.0,...,4,5,6,7,3,4,4,5,8,8


In [41]:
df_ENGHO_hogares.tail()

,id,provincia,region,subregion,trimestre,anio,pondera,cv1c04,cv1c05_a,cv1c05_b,...,qinth_t,dinth_p,dinth_r,dinth_t,qinpch_p,qinpch_r,qinpch_t,dinpch_p,dinpch_r,dinpch_t
21542,57618153,94,6,6B,3,2018,124,1,3.0,2.0,...,3,2,4,5,4,5,5,7,9,9
21543,551737923,94,6,6B,3,2018,123,1,3.0,2.0,...,4,4,7,8,5,5,5,10,10,10
21544,188896348,94,6,6B,1,2018,11,1,2.0,2.0,...,1,1,2,2,1,1,1,1,2,2
21545,127611813,94,6,6B,2,2018,68,1,3.0,3.0,...,5,9,10,10,4,5,5,8,9,10
21546,37138480,94,6,6B,2,2018,59,1,3.0,3.0,...,5,7,9,9,3,4,5,6,8,9


### Cargamos Personas:

In [42]:
df_ENGHO_personas = pd.read_csv(ENGHO_DIR/"engho2018_personas.txt", sep='|', low_memory = False)
df_ENGHO_personas.head()

,id,provincia,miembro,pondera,region,subregion,anio,trimestre,cp03,cp04,...,ibecaspub_imp,iotrosps,m_iotrosps,iotrosps_imp,itransfermon,m_itransfermon,itransfermon_imp,iautoconsumo,m_iautoconsumo,iautoconsumo_imp
0,478229345,2,1,276,1,1A,2018,4,83,1,...,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0
1,478229345,2,2,276,1,1A,2018,4,82,2,...,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0
2,824935596,2,1,271,1,1A,2018,4,75,1,...,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0
3,824935596,2,2,271,1,1A,2018,4,51,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,549385244,2,1,264,1,1A,2017,1,19,1,...,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0


### Cargamos gastos:

In [43]:
df_ENGHO_gastos = pd.read_csv(ENGHO_DIR/"engho2018_gastos.txt", sep='|', low_memory = False)
df_ENGHO_gastos.head()

,id,provincia,miembro,pondera,region,subregion,articulo,subclase,clase,grupo,division,r_imputado,cantidad,unmed,monto,forma_pago,tipo_negocio,modo_adq,lugar_adq
0,8558,14,0,749,2,2A,A0111105,A01111,A0111,A011,A01,2,1.732,K,186.19,1.0,3.0,1.0,2.0
1,8558,14,0,749,2,2A,A0111111,A01111,A0111,A011,A01,2,4.330,K,173.20,1.0,2.0,1.0,2.0
2,8558,14,0,749,2,2A,A0111111,A01111,A0111,A011,A01,2,10.825,K,433.00,1.0,3.0,1.0,2.0
3,8558,14,0,749,2,2A,A0111114,A01111,A0111,A011,A01,2,2.165,K,151.55,1.0,3.0,1.0,2.0
4,8558,14,0,749,2,2A,A0111201,A01112,A0111,A011,A01,2,2.165,K,73.61,1.0,3.0,1.0,2.0


In [44]:
df_ENGHO_gastos.tail()

,id,provincia,miembro,pondera,region,subregion,articulo,subclase,clase,grupo,division,r_imputado,cantidad,unmed,monto,forma_pago,tipo_negocio,modo_adq,lugar_adq
901799,999873115,82,0,388,2,2B,A0942101,A09421,A0942,A094,A09,2,1.0000,U,500.00,1.0,NaN,NaN,2.0
901800,999873115,82,0,388,2,2B,A1213211,A12132,A1213,A121,A12,2,465.4750,G,38.97,1.0,2.0,1.0,2.0
901801,999873115,82,0,388,2,2B,A1254101,A12541,A1254,A125,A12,2,0.3102,U,310.20,1.0,3.0,NaN,NaN
901802,999873115,82,2,388,2,2B,A0722102,A07221,A0722,A072,A07,2,23.8150,L,866.00,1.0,3.0,NaN,2.0
901803,999873115,82,3,388,2,2B,A1111217,A11112,A1111,A111,A11,1,12.9900,U,1558.80,1.0,9.0,NaN,2.0


## Dejamos listados todos los datasets descargados:

In [45]:
datasets = {
    "EPH SMVM": df_SMVM,
    "EPH RIPTE": df_RIPTE,
    "AUH Valor general": df_AUH_valor_general,
    "AUH Edad Titular": df_AUH_edad_titular,
    "IPC General": df_IPC_general_mensual,
    "IPC Canastas GBA": df_ipc_canasta_GBA,
    "IPC Canastas NEA": df_ipc_canasta_NEA,
    "IPC Canastas NOA": df_ipc_canasta_NOA,
    "IPC Canastas CUYO": df_ipc_canasta_CUYO,
    "IPC Canastas PAMPEANA": df_ipc_canasta_PAMPEANA,
    "IPC Canastas PATAGONIA": df_ipc_canasta_PATAGONIA,  
    "CBA": CBA,
    "CBT": CBT,
    "Pobreza_Indigencia": pobreza_indigencia,
    "ENGHo Hogares": df_ENGHO_hogares,
    "ENGHo Personas": df_ENGHO_personas,
    "ENGHo Gastos": df_ENGHO_gastos,
}

# <center> 3.  Revision, variables y tipos

In [46]:
for dataset, df in datasets.items():
    print(f"{dataset}: {df.shape}")

EPH SMVM: (740, 3)
EPH RIPTE: (383, 2)
AUH Valor general: (59, 5)
AUH Edad Titular: (34, 16)
IPC General: (115, 7)
IPC Canastas GBA: (109, 15)
IPC Canastas NEA: (109, 15)
IPC Canastas NOA: (109, 15)
IPC Canastas CUYO: (109, 15)
IPC Canastas PAMPEANA: (109, 15)
IPC Canastas PATAGONIA: (109, 15)
CBA: (117, 7)
CBT: (117, 7)
ENGHo Hogares: (21547, 134)
ENGHo Personas: (68725, 158)
ENGHo Gastos: (901804, 19)


In [47]:
df_ENGHO_personas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68725 entries, 0 to 68724
Columns: 158 entries, id to iautoconsumo_imp
dtypes: float64(97), int64(57), object(4)
memory usage: 82.8+ MB


### Veamos los tipos de cada dataset:

In [48]:
for dataset, df in datasets.items():
    print(f"Informacion de {dataset}: {df.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 740 entries, 3 to 742
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Mensual  740 non-null    object
 1   Diario   740 non-null    object
 2   Horario  740 non-null    object
dtypes: object(3)
memory usage: 17.5+ KB
Informacion de EPH SMVM: None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 383 entries, 0 to 382
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Período  383 non-null    object
 1   RIPTE    383 non-null    object
dtypes: object(2)
memory usage: 6.1+ KB
Informacion de EPH RIPTE: None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 5 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Periodo                            59 non-null     object
 1   Hijo                 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 901804 entries, 0 to 901803
Data columns (total 19 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            901804 non-null  int64  
 1   provincia     901804 non-null  int64  
 2   miembro       901804 non-null  int64  
 3   pondera       901804 non-null  int64  
 4   region        901804 non-null  int64  
 5   subregion     901804 non-null  object 
 6   articulo      901804 non-null  object 
 7   subclase      901804 non-null  object 
 8   clase         901804 non-null  object 
 9   grupo         901804 non-null  object 
 10  division      901804 non-null  object 
 11  r_imputado    901804 non-null  int64  
 12  cantidad      901804 non-null  float64
 13  unmed         901804 non-null  object 
 14  monto         901804 non-null  float64
 15  forma_pago    892337 non-null  float64
 16  tipo_negocio  742234 non-null  float64
 17  modo_adq      601946 non-null  float64
 18  luga

### Vemos que muchos datasets tiene los tipos mal colocados

In [49]:
for dataset, df in datasets.items():
    print(f"\n===== {dataset} =====")
    display(df.head())


===== EPH SMVM =====


8,Mensual,Diario,Horario
3,9800,392,49
4,9800,392,49
5,9800,392,49
6,9800,392,49
7,11550,462,57.75



===== EPH RIPTE =====


8,Período,RIPTE
0,1994-07-01 00:00:00,874.87
1,1994-08-01 00:00:00,893
2,1994-09-01 00:00:00,907.99
3,1994-10-01 00:00:00,910.75
4,1994-11-01 00:00:00,916.93



===== AUH Valor general =====


...

,id,provincia,region,subregion,trimestre,anio,pondera,cv1c04,cv1c05_a,cv1c05_b,...,qinth_t,dinth_p,dinth_r,dinth_t,qinpch_p,qinpch_r,qinpch_t,dinpch_p,dinpch_r,dinpch_t
0,478229345,2,1,1A,4,2018,276,4,2.0,3.0,...,4,5,7,7,3,4,4,6,8,8
1,824935596,2,1,1A,4,2018,271,4,3.0,3.0,...,4,7,8,8,4,5,5,7,9,9
2,549385244,2,1,1A,1,2017,264,4,2.0,3.0,...,2,2,3,3,3,4,4,5,7,8
3,64338930,2,1,1A,3,2018,245,4,3.0,3.0,...,5,9,9,10,5,5,5,10,10,10
4,635779227,2,1,1A,3,2018,283,4,3.0,3.0,...,4,5,6,7,3,4,4,5,8,8



===== ENGHo Personas =====


,id,provincia,miembro,pondera,region,subregion,anio,trimestre,cp03,cp04,...,ibecaspub_imp,iotrosps,m_iotrosps,iotrosps_imp,itransfermon,m_itransfermon,itransfermon_imp,iautoconsumo,m_iautoconsumo,iautoconsumo_imp
0,478229345,2,1,276,1,1A,2018,4,83,1,...,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0
1,478229345,2,2,276,1,1A,2018,4,82,2,...,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0
2,824935596,2,1,271,1,1A,2018,4,75,1,...,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0
3,824935596,2,2,271,1,1A,2018,4,51,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,549385244,2,1,264,1,1A,2017,1,19,1,...,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0



===== ENGHo Gastos =====


,id,provincia,miembro,pondera,region,subregion,articulo,subclase,clase,grupo,division,r_imputado,cantidad,unmed,monto,forma_pago,tipo_negocio,modo_adq,lugar_adq
0,8558,14,0,749,2,2A,A0111105,A01111,A0111,A011,A01,2,1.732,K,186.19,1.0,3.0,1.0,2.0
1,8558,14,0,749,2,2A,A0111111,A01111,A0111,A011,A01,2,4.330,K,173.20,1.0,2.0,1.0,2.0
2,8558,14,0,749,2,2A,A0111111,A01111,A0111,A011,A01,2,10.825,K,433.00,1.0,3.0,1.0,2.0
3,8558,14,0,749,2,2A,A0111114,A01111,A0111,A011,A01,2,2.165,K,151.55,1.0,3.0,1.0,2.0
4,8558,14,0,749,2,2A,A0111201,A01112,A0111,A011,A01,2,2.165,K,73.61,1.0,3.0,1.0,2.0


### Veamos los valores faltantes:

In [50]:
for dataset, df in datasets.items():
    print(f"\n===== Valores faltantes en {dataset} =====")
    display(df.isna().sum().sum())


===== Valores faltantes en EPH SMVM =====


0


===== Valores faltantes en EPH RIPTE =====


0


===== Valores faltantes en AUH Valor general =====


...

34989


===== Valores faltantes en ENGHo Personas =====


3339075


===== Valores faltantes en ENGHo Gastos =====


493939

#### En general solo en los datasets de ENGHo, tenemos gran cantidad de valores faltantes.

### Veamos los valores duplicados:

In [51]:
for dataset, df in datasets.items():
    print(f"\n===== Valores duplicados en {dataset} =====")
    display(df.duplicated().sum())


===== Valores duplicados en EPH SMVM =====


538


===== Valores duplicados en EPH RIPTE =====


0


===== Valores duplicados en AUH Valor general =====


...

0


===== Valores duplicados en ENGHo Personas =====


0


===== Valores duplicados en ENGHo Gastos =====


0

#### No encontramos valores duplicados en los datasets

# <center>4 . Exportamos las bases en un formato csv

In [65]:
for dataset, df in datasets.items():
    print(f"\n===== Guardando {dataset} en un csv =====")
    os.makedirs(INTERIM_DIR, exist_ok=True)
    name = dataset.replace(" ", "_").lower()
    save_csv(df, name, INTERIM_DIR)


===== Guardando EPH SMVM en un csv =====
eph_smvm

===== Guardando EPH RIPTE en un csv =====
eph_ripte

===== Guardando AUH Valor general en un csv =====
auh_valor_general

===== Guardando AUH Edad Titular en un csv =====
auh_edad_titular

===== Guardando IPC General en un csv =====
ipc_general

===== Guardando IPC Canastas GBA en un csv =====
ipc_canastas_gba

===== Guardando IPC Canastas NEA en un csv =====
ipc_canastas_nea

===== Guardando IPC Canastas NOA en un csv =====
ipc_canastas_noa

===== Guardando IPC Canastas CUYO en un csv =====
ipc_canastas_cuyo

===== Guardando IPC Canastas PAMPEANA en un csv =====
ipc_canastas_pampeana

===== Guardando IPC Canastas PATAGONIA en un csv =====
ipc_canastas_patagonia

===== Guardando CBA en un csv =====
cba

===== Guardando CBT en un csv =====
cbt

===== Guardando ENGHo Hogares en un csv =====
engho_hogares

===== Guardando ENGHo Personas en un csv =====
engho_personas

===== Guardando ENGHo Gastos en un csv =====
engho_gastos
